# Huawei Technology Lab 2 — MindSpore Medical Image Classification
Build a small CNN with Huawei MindSpore and learn why healthcare evaluation needs more than accuracy.

This compact repository edition uses synthetic 28×28 images so it runs without patient data or an external download. Replace only with approved public teaching data when appropriate.

**Educational prototype — not for clinical diagnosis or treatment.**

In [ ]:
import numpy as np
import mindspore as ms
from mindspore import nn, ops
import mindspore.dataset as ds

ms.set_seed(42)
rng = np.random.default_rng(42)

def make_images(n):
    images = rng.normal(0.25,0.08,size=(n,1,28,28)).astype(np.float32)
    labels = rng.integers(0,2,size=n,dtype=np.int32)
    yy,xx=np.mgrid[:28,:28]
    for i,label in enumerate(labels):
        if label==1:
            cx,cy=int(rng.integers(9,19)),int(rng.integers(9,19))
            mask=(xx-cx)**2+(yy-cy)**2 < int(rng.integers(10,24))
            images[i,0][mask]+=0.45
    return np.clip(images,0,1), labels

X_train,y_train=make_images(600)
X_test,y_test=make_images(160)
train_ds=ds.NumpySlicesDataset((X_train,y_train),column_names=['image','label'],shuffle=True).batch(32)
print(X_train.shape)

## Build a LeNet-style MindSpore CNN

In [ ]:
class MedCNN(nn.Cell):
    def __init__(self):
        super().__init__()
        self.features=nn.SequentialCell(
            nn.Conv2d(1,6,5,pad_mode='valid',has_bias=True),nn.ReLU(),nn.MaxPool2d(2,2),
            nn.Conv2d(6,16,5,pad_mode='valid',has_bias=True),nn.ReLU(),nn.MaxPool2d(2,2))
        self.classifier=nn.SequentialCell(nn.Flatten(),nn.Dense(16*4*4,64),nn.ReLU(),nn.Dense(64,2))
    def construct(self,x):
        return self.classifier(self.features(x))

net=MedCNN()
loss_fn=nn.CrossEntropyLoss()
optimizer=nn.Adam(net.trainable_params(),learning_rate=0.001)

def forward_fn(x,y):
    logits=net(x)
    return loss_fn(logits,y),logits
grad_fn=ms.value_and_grad(forward_fn,None,optimizer.parameters,has_aux=True)
def train_step(x,y):
    (loss,logits),grads=grad_fn(x,y)
    optimizer(grads)
    return loss

for epoch in range(5):
    losses=[]
    for x,y in train_ds.create_tuple_iterator(): losses.append(float(train_step(x,y).asnumpy()))
    print(f'Epoch {epoch+1}: {np.mean(losses):.4f}')

## Evaluate accuracy, sensitivity, specificity, precision and F1
The positive class is an artificial abnormal pattern for teaching only.

In [ ]:
net.set_train(False)
logits=net(ms.Tensor(X_test,ms.float32))
pred=ops.argmax(logits,axis=1).asnumpy()
tn=int(((y_test==0)&(pred==0)).sum()); fp=int(((y_test==0)&(pred==1)).sum())
fn=int(((y_test==1)&(pred==0)).sum()); tp=int(((y_test==1)&(pred==1)).sum())
accuracy=(tp+tn)/len(y_test)
sensitivity=tp/max(tp+fn,1)
specificity=tn/max(tn+fp,1)
precision=tp/max(tp+fp,1)
f1=2*precision*sensitivity/max(precision+sensitivity,1e-9)
print('Confusion matrix [[TN,FP],[FN,TP]]:',[[tn,fp],[fn,tp]])
print({'accuracy':round(accuracy,3),'sensitivity':round(sensitivity,3),'specificity':round(specificity,3),'precision':round(precision,3),'f1':round(f1,3)})
ms.save_checkpoint(net,'mindspore_medical_image_classifier.ckpt')

## Why accuracy is not enough
A model can appear accurate when one class dominates. In healthcare teaching, inspect false negatives and false positives and report sensitivity and specificity alongside accuracy. Benchmark success is not clinical validation.